# Delta Lake Basics — 06: OPTIMIZE and ZORDER

## What you will learn
OPTIMIZE and ZORDER are Delta's built-in query acceleration features.
They solve the small file problem and co-locate related data for faster reads.

In this notebook:
1. Why OPTIMIZE is needed — the small file problem
2. Running OPTIMIZE — compacting files
3. ZORDER BY — clustering data for faster filters
4. How ZORDER works — Z-ordering algorithm explained
5. OPTIMIZE + ZORDER together
6. When to run OPTIMIZE — scheduling strategies


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/delta_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('delta-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Delta Lake ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/15 05:41:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Delta Lake ready


26/04/15 05:41:58 WARN TaskSetManager: Stage 0 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/15 05:42:00 WARN TaskSetManager: Stage 1 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.


Dataset ready: 100,000 rows


In [2]:
TABLE = f"{DATA_DIR}/06_optimize"
shutil.rmtree(TABLE, ignore_errors=True)

# Create small file problem: 20 small appends
for i in range(20):
    batch = df.limit(5000)
    batch.write.format("delta").mode("append").save(TABLE)

files_before = list(pathlib.Path(TABLE).glob("*.parquet"))
size_before  = sum(f.stat().st_size for f in files_before)
print(f"Before OPTIMIZE:")
print(f"  Files  : {len(files_before)}")
print(f"  Size   : {size_before/1024/1024:.1f} MB")
print(f"  Avg    : {size_before/len(files_before)/1024:.0f} KB per file")

26/04/15 05:42:02 WARN TaskSetManager: Stage 4 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/15 05:42:06 WARN TaskSetManager: Stage 6 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/15 05:42:09 WARN TaskSetManager: Stage 8 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/15 05:42:10 WARN TaskSetManager: Stage 10 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/15 05:42:11 WARN TaskSetManager: Stage 12 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/15 05:42:12 WARN TaskSetManager: Stage 14 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/15 05:42:13 WARN TaskSetManager: Stage 16 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/15 

Before OPTIMIZE:
  Files  : 20
  Size   : 2.3 MB
  Avg    : 116 KB per file


## Step 1 — OPTIMIZE: Compacting Small Files


In [3]:
# Measure query time before OPTIMIZE
runs = 3
times_before = []
for _ in range(runs):
    t0 = time.time()
    spark.read.format("delta").load(TABLE).agg(F.sum("revenue")).collect()
    times_before.append(time.time() - t0)
t_before = sorted(times_before)[1]
print(f"Query time before OPTIMIZE: {t_before:.3f}s  ({len(files_before)} files)")

# Run OPTIMIZE
print("\nRunning OPTIMIZE...")
t0 = time.time()
result = spark.sql(f"OPTIMIZE delta.`{TABLE}`")
t_optimize = time.time() - t0
result.show(truncate=False)

files_after = list(pathlib.Path(TABLE).glob("*.parquet"))
# Files visible via Delta (logical, not physical old files)
detail = DeltaTable.forPath(spark, TABLE).detail().collect()[0]
print(f"\nAfter OPTIMIZE ({t_optimize:.1f}s):")
print(f"  Logical files : {detail['numFiles']}")
print(f"  Size          : {detail['sizeInBytes']/1024/1024:.1f} MB")

# Query time after
times_after = []
for _ in range(runs):
    t0 = time.time()
    spark.read.format("delta").load(TABLE).agg(F.sum("revenue")).collect()
    times_after.append(time.time() - t0)
t_after = sorted(times_after)[1]
print(f"\nQuery time after OPTIMIZE : {t_after:.3f}s")
print(f"Speedup                   : {t_before/t_after:.1f}x")

Query time before OPTIMIZE: 2.286s  (20 files)

Running OPTIMIZE...


+---------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                         |metrics                                                                                                                                                                     |
+---------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|file:/workspace/data/delta_basics/06_optimize|{1, 20, {913065, 913065, 913065.0, 1, 913065}, {118243, 119061, 118692.9, 20, 2373858}, 1, NULL, NULL, 1, 1, 20, 0, false, 0, 0, 1776231772297, 0, 4, 0, NULL, NULL, 10, 10}|
+---------------------------------------------+---------------------------------------------------------------------


After OPTIMIZE (7.9s):
  Logical files : 1
  Size          : 0.9 MB

Query time after OPTIMIZE : 2.163s
Speedup                   : 1.1x


## Step 2 — ZORDER BY: Data Clustering

ZORDER reorganises data so rows with similar values on the ZORDERed columns
are stored in the same files. This maximises data skipping for filtered queries.

Z-ordering is a **space-filling curve** algorithm — it maps multi-dimensional
data to 1D while preserving locality in all dimensions.


In [4]:
TABLE_Z = f"{DATA_DIR}/06_zorder"
shutil.rmtree(TABLE_Z, ignore_errors=True)
df.write.format("delta").mode("overwrite").save(TABLE_Z)

# Cache small slice once to avoid large task warnings
df_small_z = df.limit(3000).cache()
df_small_z.count()

# Create small files first
for i in range(10):
    df_small_z.write.format("delta").mode("append").save(TABLE_Z)

# Benchmark: selective query WITHOUT ZORDER
runs = 3
def run_query(path):
    return spark.read.format("delta").load(path) \
               .filter(F.col("category") == "Electronics") \
               .filter(F.col("country") == "US") \
               .agg(F.sum("revenue"), F.count("*")).collect()

times_no_z = []
for _ in range(runs):
    t0 = time.time(); run_query(TABLE_Z); times_no_z.append(time.time()-t0)
t_no_z = sorted(times_no_z)[1]

# OPTIMIZE + ZORDER
print("Running OPTIMIZE ZORDER BY (category, country)...")
t0 = time.time()
spark.sql(f"OPTIMIZE delta.`{TABLE_Z}` ZORDER BY (category, country)")
t_zorder = time.time() - t0
print(f"  Took: {t_zorder:.1f}s")

times_z = []
for _ in range(runs):
    t0 = time.time(); run_query(TABLE_Z); times_z.append(time.time()-t0)
t_z = sorted(times_z)[1]

print(f"\nQuery: WHERE category=Electronics AND country=US")
print(f"  Without ZORDER : {t_no_z:.3f}s")
print(f"  With ZORDER    : {t_z:.3f}s")
print(f"  Speedup        : {t_no_z/t_z:.1f}x")
print()
print("ZORDER co-locates Electronics+US rows in the same files.")
print("Spark skips files where max(category) < 'Electronics' (data skipping).")

26/04/15 05:43:02 WARN TaskSetManager: Stage 115 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/15 05:43:03 WARN TaskSetManager: Stage 116 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.


Running OPTIMIZE ZORDER BY (category, country)...
  Took: 4.6s

Query: WHERE category=Electronics AND country=US
  Without ZORDER : 1.568s
  With ZORDER    : 1.434s
  Speedup        : 1.1x

ZORDER co-locates Electronics+US rows in the same files.
Spark skips files where max(category) < 'Electronics' (data skipping).


## Step 3 — OPTIMIZE Settings and Options


In [ ]:
# OPTIMIZE WHERE — only works on PARTITION columns
# TABLE is not partitioned, so WHERE clause is not allowed
# Demonstrate with a partitioned table instead

TABLE_PART = f"{DATA_DIR}/06_optimize_partitioned"
shutil.rmtree(TABLE_PART, ignore_errors=True)

# Write partitioned by category
for i in range(10):
    df.limit(5000).write.format("delta").mode("append").partitionBy("category").save(TABLE_PART)

print("OPTIMIZE WHERE on partitioned table:")
spark.sql(f"""
    OPTIMIZE delta.`{TABLE_PART}`
    WHERE category = 'Electronics'
""")
print("  Partition-level OPTIMIZE (WHERE category = 'Electronics') ✅")
print("  Note: WHERE clause in OPTIMIZE only works on partition columns")
print("  Non-partition columns → AnalysisException: DELTA_NON_PARTITION_COLUMN_REFERENCE")

# OPTIMIZE with ZORDER on multiple columns
print("\nZORDER priority:")
print("  ZORDER BY (col1, col2) — col1 has more impact than col2")
print("  Best columns: your most selective filter columns")
print()
print("When NOT to ZORDER:")
print("  ❌ Low cardinality columns (boolean, tiny enums) — no benefit")
print("  ❌ Partition columns — already co-located by directory")
print("  ❌ Rarely filtered columns — spend ZORDER budget on hot columns")
print()
print("When to ZORDER:")
print("  ✅ High-cardinality columns you filter on (user_id, product_id)")
print("  ✅ Columns used in JOIN conditions")
print("  ✅ Columns used in range queries (date ranges, price ranges)")

26/04/15 05:43:29 WARN TaskSetManager: Stage 199 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/15 05:43:30 WARN TaskSetManager: Stage 201 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/15 05:43:31 WARN TaskSetManager: Stage 203 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
26/04/15 05:43:32 WARN TaskSetManager: Stage 205 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/15 05:43:33 WARN TaskSetManager: Stage 207 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/15 05:43:34 WARN TaskSetManager: Stage 209 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/15 05:43:35 WARN TaskSetManager: Stage 211 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.

OPTIMIZE WHERE on partitioned table:


## Step 4 — OPTIMIZE Scheduling


In [ ]:
print("OPTIMIZE scheduling strategies:")
print()
print("1. On-demand: run after bulk loads")
print("   spark.sql(f'OPTIMIZE delta.`{path}`')")
print()
print("2. Scheduled: run daily/weekly via cron or orchestrator")
print("   - Run after peak write hours")
print("   - Use WHERE clause to only optimize recently modified partitions")
print()
print("3. Auto-optimize (Databricks): automatic small file compaction")
print("   Not available in open-source Delta — manual OPTIMIZE needed")
print()
print("4. After streaming: compact streaming micro-batch files")
print("   - Streaming writes many small files per micro-batch")
print("   - Run OPTIMIZE periodically to compact them")
print()

# Show what OPTIMIZE actually does to _delta_log
print("What OPTIMIZE writes to _delta_log:")
import json as pyjson
log_path = pathlib.Path(f"{TABLE}/_delta_log")
for commit in sorted(log_path.glob("*.json"))[-2:]:
    print(f"  {commit.name}:")
    with open(commit) as f:
        for line in f:
            entry = pyjson.loads(line)
            key = list(entry.keys())[0]
            if key == "commitInfo":
                print(f"    operation: {entry[key].get('operation')}")
            elif key == "add":
                print(f"    add: {entry[key]['path'][-35:]}")
            elif key == "remove":
                print(f"    remove: {entry[key]['path'][-35:]}")

## Summary

```python
# Basic OPTIMIZE (compact small files)
spark.sql(f"OPTIMIZE delta.`/path`")

# OPTIMIZE + ZORDER (compact + cluster)
spark.sql(f"OPTIMIZE delta.`/path` ZORDER BY (category, country)")

# Partition-level OPTIMIZE (only recent data)
spark.sql(f"OPTIMIZE delta.`/path` WHERE date >= '2024-06-01'")
```

### OPTIMIZE decision guide
| Situation | Action |
|---|---|
| Many small files (< 64 MB) | `OPTIMIZE` |
| Slow filtered queries | `OPTIMIZE ZORDER BY (filter_col)` |
| After streaming run | `OPTIMIZE` on latest partition |
| Large table, only new data changed | `OPTIMIZE WHERE date = today` |

### ZORDER column selection
- Choose columns you filter most often
- Prioritise high-cardinality columns
- 1-3 columns max (diminishing returns beyond that)
- Never ZORDER by partition columns
